
# 오늘 실습 내용에 대한 배경 설명

우선, 우리가 학습된 모델을 저장할 때 사용하는 파일 확장자부터 살펴보겠습니다.

1. `.bin` — PyTorch 모델 포맷<br>
2. `.safetensors` — Hugging Face에서 사용하는 보안 강화 모델 포맷<br>
3. `.gguf` — `llama.cpp` 라이브러리용 경량 LLM 모델 포맷<br>
4. `.tflite` — TensorFlow Lite용 임베디드·모바일 환경 최적화 포맷<br>
<br>
1번과 2번은 이전에 Hugging Face 모델에서 확인해본 포맷들이었다면, <br>
오늘은 GGUF와 TFLite 포맷, 특히 GGUF 포멧에 대해 알아보겠습니다.


<br>


<br>


---

<br>

## 모델 경량화와 양자화

모델의 경량화(Model Optimization)란
모델의 크기와 연산량을 줄이는 기술.<br>
이중 대표적인 방법이 바로 양자화(Quantization)입니다.
<br>
<br>
양자화를 통해 모델의 가중치(Weights) 정밀도를 줄이면<br>
모델의 **크기** 그리고 **메모리 사용량**, 그리고 **연산량**이 크게 **감소**합니다.<br>
그리고 모델의 양자화를 통해 모델을 .gguf 포맷이나 .tflite 포맷으로 변환할 수 있습니다.
<br>

---
<br>

## 왜 양자화를 할까?

우리는 일반적으로
파운데이션 모델 → 파인튜닝(추가 학습) → 추론 서버 배포
의 단계를 거칩니다.

이때, 파인튜닝된 모델을 비트 수를 줄여서 추론을 진행하면 서버의 GPU 메모리와 연산 부담을 줄일 수 있습니다.

<br>

---

<br>

## 모델 정밀도(비트 수)의 차이

일반적으로 모델의 파라미터(가중치)는 float32 (FP32) 정밀도로 저장되고 <br>
학습 시에는 GPU 메모리를 절약하기 위해 float16 (FP16) 또는 bfloat16 (BF16)을 자주 사용합니다.

<br>
그러나 양자화를 적용하면
가중치 (파라미터)를 4bit 정수 또는 NF4 (normalized float4) 형태로 표현하여
모델 크기와 연산량을 더욱 줄일 수 있습니다.
<br>
단, 정밀도가 낮아질수록 일부 연산 정확도는 다소 떨어질 수 있습니다.
(trade-off 관계)

<br>
<br>

---

<br>


## 그런데 Hugging Face에서 이미 양자화된 모델을 받으면 되지 않나요?

좋은 질문입니다.
하지만 정답은 ‘아니요’입니다.
배포 단계에서 한 번 더 양자화를 해야 합니다.

그 이유를 살펴보면 다음과 같습니다.


<br>


---


<br>


## Hugging Face의 모델 종류

Hugging Face에는 용도에 따라 다양한 형태의 모델이 공개되어 있습니다.

* **베이스 모델 (Base Model)**
  예: `meta-llama/Llama-2-7b`
  → 사전학습(pretrained)된 원본 모델 (보통 FP16 또는 BF16 정밀도)

* **양자화된 경량 모델 (Quantized Inference Model)**
  예: `TheBloke/Llama-2-7B-GGUF`, `google/flan-t5-int8`
  → 추론용으로 가볍게 만들어진 모델 (학습용이 아님)

예시 > https://huggingface.co/Qwen/Qwen3-0.6B

즉,
Hugging Face의 양자화 모델은 대부분 “배포(추론)” 목적이지,
“학습(파인튜닝)”을 위한 형태가 아닙니다.

<br>



---

<br>

## QLoRA와 학습 단계의 양자화

그럼 우리가 파인튜닝할 때 사용했던 양자화는 무엇일까요?

그것이 바로 QLoRA입니다.
1. QLoRA는 훈련 중 GPU 메모리 절약을 위해 모델을 4bit 상태로 로드(load) 한 다음 <br>
2. 원래 모델의 4bit 가중치는 업데이트하지 않고 고정, 그리고 <br>
3. LoRA 어댑터(추가 학습 파라미터) 만 학습 했습니다.

하지만, 실제 저장되는 학습 결과(LoRA 어댑터)는 여전히 `float16` 혹은 `float32` 형식입니다.

```python
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-2-7b",
    load_in_4bit=True,  # 학습 시 연산 효율을 위한 4bit 로드
    device_map="auto"
)
```

즉,
* QLoRA의 4bit는 연산 효율을 위한 임시 정밀도이며
* 파인튜닝의 결과물은 여전히 양자화되지 않은 float 계열 모델입니다.

<br>

---

<br>

우리는 양자화 단계를 구분해서 이해를 해야 합니다.

<br>


## 학습용 양자화 vs 배포용 양자화

| 구분               | 목적           | 결과물 형태                |
| ---------------- | ------------ | --------------------- |
| QLoRA (학습 중)     | GPU 메모리 절감   | LoRA 어댑터 (float16/32) |
| 배포용 양자화 (GGUF 등) | 추론 효율, 속도 향상 | int4 / int8 정수형 모델    |

<br>

* QLoRA의 양자화는 훈련 효율을 위한 것이고,
* GGUF 등의 양자화는 추론 효율을 위한 것입니다.

<br>

---

<br>

결국 모델을 배포하는 전체 과정을 살펴보면 다음과 같습니다.

## 모델 배포 전체 과정

```
[1] Hugging Face 베이스 모델
        ↓
[2] 파인튜닝 (내 데이터로 추가 학습)
        ↓
[3] 파인튜닝 완료 모델 (예: FP16)
        ↓
[4] GGUF 변환 (포맷 변환 + 양자화 포함)
        ↓
[5] llama.cpp 로드
        ↓
[6] C++ 기반 고속 추론 (배포)
```

<br>

---

<br>


## llama.cpp와 GGUF 포맷

파인튜닝한 모델을 추론 서버에 올렸다고 가정해봅시다. <br>
이제 추론을 실행하면 기본적으로 PyTorch가 모델을 로드해 동작하지만, <br>
이 방식은 GPU 자원을 많이 소모하고 속도가 느립니다. <br>

이를 해결하기 위해 사용하는 것이 바로 <br>
C/C++ 기반 고속 추론 엔진인 `llama.cpp` 입니다.

<br>

---

<br>

### llama.cpp의 특징

* GGUF 포맷을 사용하는 다양한 모델(LLaMA, Qwen 등)을 지원
* Windows / macOS / Linux / Android / iOS 등 멀티플랫폼 지원
* CPU·GPU 모두에서 실행 가능
* GGUF 포맷 모델은 GPU가 없어도 CPU만으로 효율적인 추론이 가능
  * Q4_K_M 등 경량 양자화 모델을 사용하면 고성능 CPU 기준, 준실시간 추론 속도 가능

<br>

---

<br>

## 배포 단계 요약

```
[1] Hugging Face 베이스 모델
        ↓
[2] 파인튜닝 (QLoRA 등)
        ↓
[3] 파인튜닝 결과 (FP16)
        ↓
[4] GGUF 변환 (양자화 포함)
        ↓
[5] llama.cpp 로드
        ↓
[6] C++ 기반 고속 추론
```

<br>

이 과정을 거치면 CPU 최적화 환경이나 모바일(Android/iOS)에서도 <br>
경량화된 AI 모델을 로컬에서 직접 실행할 수 있습니다.

<br>

---

<br>

## PyTorch vs llama.cpp

| 항목      | PyTorch                | llama.cpp        |
| ------- | ---------------------- | ---------------- |
| 주 사용 목적 | 연구, 실험, 학습             | 추론 및 배포          |
| 포맷      | .bin, .safetensors | .gguf          |
| 언어      | Python 기반              | C/C++ 기반         |
| 실행 환경   | GPU 중심                 | CPU/GPU/모바일      |
| 특징      | 개발 친화적            | 고속, 경량 |

<br>

따라서,
연구·개발 단계에서는 PyTorch를 사용하고,
배포·운영 단계에서는 llama.cpp를 사용하는 것이 일반적입니다.

<br>

---

<br>

자, 이제 이 개념들을 기반으로 실습 노트북을 열고 직접 실행해 보겠습니다.

<br>
